In [100]:
import os
import re
import pandas as pd

In [102]:
os.makedirs('processed', exist_ok=True)

In [104]:
def map_target_lines(code, target_lines):
    # Split the code into lines.
    original_lines = code.splitlines()
    new_lines = []
    mapping = {}  # mapping from original line number to new line number

    # Process each original line (1-indexed).
    for i, line in enumerate(original_lines, start=1):
        if line.strip():  # if the line is not empty (or only whitespace)
            new_lines.append(line)
            mapping[i] = len(new_lines)  # record new line number
        else:
            mapping[i] = None  # line is removed (empty)

    # For each target line, get the corresponding new line number.
    new_target_lines = {}
    for orig in target_lines:
        new_num = mapping.get(orig)
        if orig <= len(original_lines):
            original_line = original_lines[orig - 1] # 0 - index, since orig is 1-indexed
            if re.match(r"^\s*\*", original_line) or re.match(r"^\s*/\*", original_line) or re.match(r"^\s*//", original_line) or re.match(r"^\s*\*/", original_line):
                new_target_lines[orig] = "comment line!"
            else: 
                if new_num is not None:
                    new_target_lines[orig] = new_num
                else:
                    # Optionally, you might want to handle cases where the target line is empty.
                    new_target_lines[orig] = "Removed (empty line)"
    return new_lines, new_target_lines

In [106]:
chart = 'defects4j_original/Chart'
closure = 'defects4j_original/Closure'
lang = 'defects4j_original/Lang'
math_p = 'defects4j_original/Math'
mockito = 'defects4j_original/Mockito'
time = 'defects4j_original/Time'

new_rows = []
total_files = 0
index = 0
for path in [chart, closure, lang, math_p, mockito, time]:
    folders = [f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))]
    for folder in folders:
        java_file = f'{path}/{folder}/b{folder}.java'
        metadata_file = f'{path}/{folder}/metadata.json'

        if os.path.exists(java_file) and os.path.exists(metadata_file):
            total_files += 1
            with open(java_file, 'r', encoding='utf-8', errors='ignore') as f:
                java_code = f.read()
                lines = java_code.split('\n')
                num_lines = len(lines)

            # Read metadata and extract bug line numbers
            with open(metadata_file, 'r', encoding='utf-8') as f:
                metadata = json.load(f)
                bug_lines = metadata.get("bug_line_number", [])

            if len(lines) > 0 and len(bug_lines) > 0:
                new_lines, new_target_lines = map_target_lines(java_code, bug_lines)
                if len(new_lines) > 2048:
                    pass
                    # print(f"Maximum allowed number of lines exceeded: {java_file}: {num_lines} lines")
                else:
                    new_source_code = '\n'.join(new_lines)

                    new_line_numbers_for_new_code = []
                    for orig, new in new_target_lines.items():
                        if isinstance(new, int):
                            new_line_numbers_for_new_code.append(new)
                        else:
                            # print("Something wrong with line mapping after code compression by removing lines with whitespaces", new)
                            continue
                    new_line_numbers_for_new_code.sort()
                
                    zero_indexed_target_lines = []
                    for e in new_line_numbers_for_new_code:
                        if e - 1 >= 0:
                            zero_indexed_target_lines.append(e - 1)
                        else:
                            raise Exception("Non-continuable error occurred!!!") 
                
                    if len(new_line_numbers_for_new_code) > 0:
                        row = {'index': index, 'source_code': new_source_code, 'vuln_lines': str(zero_indexed_target_lines)}
                        new_rows.append(row)
                        index += 1
                    else:
                        pass
                        # print(f"Something wrong with the source_code in index {index}")
        else:
            print(f"Missing in {path}/{folder}:")
            if not os.path.exists(java_file):
                print(f"  - Java file missing: {java_file}")
            if not os.path.exists(metadata):
                print(f"  - Metadata missing: {metadata}")
print(f"Total number of files: {total_files}")
print(f"No issue number of files: {index}")

df = pd.DataFrame(new_rows)
df.to_csv("processed/defects4j.csv", index=False)

Total number of files: 385
No issue number of files: 352


In [107]:
df

,index,source_code,vuln_lines
0,0,/* ===========================================...,[1657]
1,1,/* ===========================================...,[60]
2,2,/* ===========================================...,[263]
3,3,/* ===========================================...,[130]
4,4,/* ===========================================...,[427]
...,...,...,...
347,347,/*\n * Copyright 2001-2010 Stephen Colebourne...,"[1536, 1537, 1538, 1539, 1541, 1542]"
348,348,/*\n * Copyright 2001-2013 Stephen Colebourne...,"[179, 182, 183, 864, 865, 885, 886]"
349,349,/*\n * Copyright 2001-2011 Stephen Colebourne...,"[665, 667]"
350,350,/*\n * Copyright 2001-2013 Stephen Colebourne...,"[264, 267, 268, 269, 271]"
